## client HTTP/REST en Python avec requests

In [ ]:
# ------------------------ utiliser les fonctions globales (get(), post(), ...) ---------------

import requests

URL = "https://gorest.co.in/public/v3/users"

# requests.(<get>,<post>,<put>,<patch>,<delete>,<head> on la même interface !!! params, data, files, headers, url, ...)
r = requests.get(URL, params={"page[size]": 5}, timeout=10, verify=False)

print(r.status_code)                    # 200
# CaseInsensitiveDict génère des clés uniques mais insensibles à la casse ==> "Content-Type" == "content-type"
print(r.headers["Content-Type"])        # application/json...
print(r.url)                            # URL réellement envoyée
print(r.elapsed.total_seconds())        # durée de l'échange

# En HTTP/REST le corpts de réponse devrait être soit application/json ou application/xml à la rigueur
if "application/json" in r.headers["Content-Type"]:
    # transforme le corps convertible en json étant donné le header , en list[dict] ou dict
    corps = r.json()
    for u in corps["data"]:
        print(u["id"], u["name"], u["status"])
    print(corps["meta"])                    # has_more, next_cursor, total

### alternatives
# si content-type == text/html ou application/xml ou text/plain ==> r.text ( str CHARSET Utf-8)
# si content-type == application/zip, video/mp4, application/octet-stream ==> r.content ( bytes )


In [ ]:
# -------------------------- RAPPEL sur getattr,  setattr, delattr, hasattr ----------

class Truc:
    pass

t = Truc()
print("attribut param ? ", hasattr(t, "param"))
setattr(t, "param", 10)
print("attribut param ? ", hasattr(t, "param"))
print(getattr(t, "param"))
delattr(t, "param")
print("attribut param ? ", hasattr(t, "param"))

print("-"*10 + "avec n'importe quoi objet comme requests" + "-"*10)
method = input("méthode http ?")
# méthode dynamique
getattr(requests, method)


In [ ]:
# ---------------------------------- idem avec la classe de base Request -----------------------
import requests
import urllib3 # dépendance tierce utilisée par requests

urllib3.disable_warnings() # enlever les warnings du verify

## REM: Request a la même signature que les fonctions globales mais rajoute la méthode

req = requests.Request(
    "GET",
    "https://gorest.co.in/public/v3/users",
    params={"filter[status]": "active", "sort": "name", "page[size]": 3},
    headers={"Accept": "application/json"},
)
prepared = req.prepare()

print(prepared.method, prepared.url)
print(dict(prepared.headers))
print(prepared.body)           # None pour un GET

##### pour envoyer la requête préparée : on doit créer un objet requests.Session(...)
s = requests.Session()
### ... envoyer la requête préparée avec .send()
r = s.send(prepared, allow_redirects=False, verify=False)

# if r.ok
if 200 <= r.status_code < 300:
    if "application/json" in r.headers["Content-Type"]:
        corps = r.json()
        for u in corps["data"]:
            print(u["id"], u["name"], u["status"])
        print(corps["meta"])
    else:
        print(f"mauvais content-type {r.headers["Content-Type"]} !")
else:
    print(f"mauvais status {r.status_code} !")



In [ ]:
# ------------------------ sécuriser les paramètres (injections) --------------------
from urllib.parse import quote # urllib est auto inclus avec python

# FIXES: protocole, domaine, radical, version
BASE = "https://gorest.co.in/public/v3"

# ici on fixe la ressource mais on pourrait aussi la paramétrer

def url_utilisateur(user_id: str) -> str:
    """
    retourne l'url d'un endpoint en utilisant quote qui peut échapper les paramètres de chemin ou de requêtes
    ICI: endpoint pour UN SEUL id => pas de query params => safe="" neutralise tout
    """
    return f"{BASE}/users/{quote(user_id, safe='')}"


# injection possible: /
r = requests.get(url=BASE + '/019d00aa/b240-7d95-8cc2-479f04f365d7')
print(r.url, r.json())

# / urlencodé donc neutralisé
url = url_utilisateur('019d00aa/b240-7d95-8cc2-479f04f365d7')
print(url)
r = requests.get(url=url)
print(r.url, r.json())


In [ ]:
# ------------------- inejections de configuration avec @dataclass et load_dotenv --------------
from dotenv import load_dotenv
import os
from dataclasses import dataclass, field

# charge les variable du fichier .env à la racine du projet en tant que variable d'env pour le processus courant
load_dotenv()

# print(os.getenv('GOREST_BASE_URL'), os.getenv('GOREST_TOKEN'))

# décorateur paramétré qui tranforme la classe en objet de données en ajoutant un __init__ mais customisable !!
# ce __init__ nécessite des annotations (contrôle de type !!)
@dataclass(
    frozen=True,  # ce __init__ retourne un objet immuable
    kw_only=True # les attributs ne peuvent appelés que de façon nommée (*, param="xx")
)
class Settings:
    base_url: str = ""
    token: str | None = field(default=None, repr=False) # repr: non lisible dans les logs !!
    timeout: tuple[float, float] = (3.05, 10.0) # (connexion, lecture) en secondes

    @classmethod
    def form_env(cls):
        """méthode de classe à utilier directement avec Settings.from_env(...)"""
        return cls(
            base_url=os.getenv('GOREST_BASE_URL'),
            token=os.getenv('GOREST_TOKEN') or None
        )

obj = Settings.form_env()
print(obj) # on ne voit pas le secret
# obj.token = ... # ERROR

In [ ]:
# ---------------- gestion d'entête Bearer côté client et réutilisable!!
from dotenv import load_dotenv
import os
import requests


load_dotenv()

token = os.getenv("GOREST_TOKEN")

# 1. En-tête à la main
# r = requests.get(URL, headers={"Authorization": f"Bearer {token}"}, timeout=10)

#### 2. Objet d'authentification réutilisable
class BearerAuth(requests.auth.AuthBase):
    def __init__(self, token: str):
        self.token = token

    def __call__(self, r: requests.PreparedRequest) -> requests.PreparedRequest:
        """
        l'objet BearAuth(token) doit être lui même appelable !!!
        auth = BearAuth(token)
        auth(...)
        """
        r.headers["Authorization"] = f"Bearer {self.token}"
        return r

URL = "https://gorest.co.in/public/v3/users"

# r = requests.post(
#     URL,
#     auth=BearerAuth(token), 
#     json={"name":"matt2 LAMAM2","email":"mlamamra2@example.com"}
# )
# print(r.status_code, r.json())

# # les données crées sont privées donc le GET demande aussi le jeton !!
# r = requests.get(
#     f'{URL}/{r.json()["id"]}', 
#     auth=BearerAuth(token)
# )
# print(r.status_code, r.json())

### 3. Session : l'authentification s'applique à tous les appels
# s = requests.Session()

# les sessions utilisent les gestionnaires de contexte
with requests.Session() as s:
    s.auth = BearerAuth(token)

    r = s.post(URL, timeout=10)
    r = s.get(f"{URL}/{r.json()["id"]}", timeout=10)
# s.close()



In [ ]:
# --------------- RAPPEL sur l'ellipse ... ----------------
from typing import Iterable, Any, Callable

def func(*params: tuple[str, ...]):
    """
    annotation précise d'un * qui n'utilise que des str
    """
    return params


# -------------------> params, -> retour
def my_map(f: Callable[...,      str], data: Iterable[Any]):
    return [ f(d) for d in data ]

In [ ]:
# --------------------------- gestion d'erreur avec requests ---------------------
import requests
from requests.exceptions import ConnectionError, Timeout, HTTPError


URL_network = "https://gorrrrrrest.co.in/public/v3/users"
URL_404 = "https://gorest.co.in/public/v3/users/xxxxxxxxx"
URL = "https://gorest.co.in/public/v3/users"

try:
    # r = requests.get(URL_network, timeout=(3.05, 10))
    r = requests.get(URL_404, timeout=(3.05, 10))
    # r = requests.get(URL, timeout=(3.05, 10))
    # par défaut les 4XX et 5XX ne déclenchent pas d'exception
    # if not (200 <= r.status_code < 300):
    #     print(r.status_code, r.json())
    
    # pour déclencher explicetement une exception en cas de statuts aberrants
    r.raise_for_status()
except (ConnectionError, Timeout) as e: # réseau indisponible OU trop lent : réessayer ou abandonner
    print(e)   # ... remplace pass
except HTTPError as exc:
    """
    HTTPError: exception générale pour capturer les pb de statuts
    """
    print(exc.response.status_code, exc.response.json())


In [ ]:
# ---------------------- idem avec erreur personnalisée standardisée RFC 9457 ------------------------
import requests, uuid
from requests.exceptions import ConnectionError, Timeout, HTTPError

class ApiError(Exception):
    """Réponse d'erreur du service (corps Problem Details si présent)."""

    def __init__(self, status: int, problem: dict | None = None):
        self.status = status
        self.problem = problem or {}
        self.code = self.problem.get("code")
        self.errors = self.problem.get("errors", [])
        self.request_id = self.problem.get("request_id")
        detail = self.problem.get("detail") or self.problem.get("title") or "erreur HTTP"
        super().__init__(f"{status} {self.code or ''} {detail}".strip())


class AuthError(ApiError): ...          # 401, 403
class NotFound(ApiError): ...           # 404
class ValidationFailed(ApiError): ...   # 422
class RateLimited(ApiError): ...        # 429

_PAR_STATUT: dict[int, type[ApiError]] = {
    401: AuthError, 403: AuthError, 404: NotFound,
    422: ValidationFailed, 429: RateLimited,
}


def error(response: requests.Response):
    # -------------------> erreur connue / inconnue
    return _PAR_STATUT.get(r.status_code, ApiError)(r.status_code, r.json())

URL_network = "https://gorrrrrrest.co.in/public/v3/users"
URL_404 = f"https://gorest.co.in/public/v3/users/{uuid.uuid1()}"
URL = "https://gorest.co.in/public/v3/users"

try:
    # r = requests.get(URL_network, timeout=(3.05, 10))
    r = requests.get(URL_404, timeout=(3.05, 10))
    # r = requests.get(URL, timeout=(3.05, 10))
    # par défaut les 4XX et 5XX ne déclenchent pas d'exception
    if not (200 <= r.status_code < 300):
        raise error(r)
except (ConnectionError, Timeout) as e: # réseau indisponible OU trop lent : réessayer ou abandonner
    print(e)   # ... remplace pass
except ApiError as e:
    """
    type d'erreur ApiError
    """
    print(e, type(e).__name__, e.problem)

In [ ]:
# ----------------- ajouter un retry en cas de certaines d'erreurs --------------------

from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry # voir également la librairie tenacity plus générique (tous protocoles...)


URL_404 = f"https://gorest.co.in/public/v3/users/{uuid.uuid1()}"

retry = Retry(
    total=3,                                          # 3 reprises au maximum
    backoff_factor=0.5,    # attente croissante entre les reprises: b * 2 ** (k-1) k: tentative (1, 2, 3 ...)
    backoff_jitter=0.5,   # fraction aléatoire pour éviter l'effet de troupeau en cas de multi threading , async ...
    status_forcelist=(429, 404, 500, 502, 503, 504),       # codes jugés transitoires (404 test)
    allowed_methods=frozenset({"GET", "HEAD", "OPTIONS"}),   # méthodes idempotentes uniquement
    respect_retry_after_header=True,                  # honore l'entête Retry-After: xs donné par le serveur qui prends le pas de calcul du backoff automatique
    raise_on_status=False,                            # rend la dernière réponse en erreur (4XX 5XX) au lieu de lever MaxRetryError
)

with requests.Session() as s:
    s.mount("https://", HTTPAdapter(max_retries=retry))
    r = s.get(URL_404)
    print(r.raw.retries) # total= tentatives restantes
        




In [ ]:
# --------------------- paginer sur un endpoint selon les règle d'un serveur -----------------

# possibilités selon le serveur: 
# page=(1, 2, n)&limit=x&offset=y ==> condition d'arrêt ? 
# ici 1/ response.json()["meta"]["has_mode"] = True | False
#        response.json()["meta"]["cursor"] = "uuid()" | None
#        règle page[after] = meta["cursor"] pour incrémenter  # Keyset
#     2/ utiliser l'entête de réponse Link qui contient l'url suivante RFC 8288
import requests

data = []
params = {"page[size]": 5}

URL = "https://gorest.co.in/public/v3/users"
with requests.Session() as s:
    while True:
        r = s.get(URL, params=params, timeout=(3.05, 10))
        r.raise_for_status()
        corps = r.json()
        data += corps["data"]
        if not corps["meta"]["has_more"]:
            break
        params["page[after]"] = corps["meta"]["next_cursor"]

print(data)



In [ ]:
# ----------------- RAPPEL : générateurs --------------------
from sys import getsizeof

def my_range(p1: int, p2: int=0, p3: int=1):
    i = p2 if p1 > p2 and p3 > 0 else p1
    while True:
        yield i
        i += 1
        if (i > p1 and p1 > p2 and p3 > 0) or (i > p2 and p2 > p1 and p3 > 0):
            return

g = my_range(2, 5)
# valeur de retour d'une fonction utlisant yield est un objet générateur
# l'objet générateur est un itérable/itérateur
# à chaque itération: le bloc s'arrête à la prochaine instruction yield et MEMORISE l'instruction sur laquelle elle s'arrête
# return ici termine l'exécution du bloc (comme un break)
print(g)

for i in g:
    print(i)


## intérêt n°1 des générateurs
g, l, r = my_range(1000), list(range(1000)), range(1000)
print(getsizeof(g), getsizeof(l), getsizeof(r))


def yield_from_example(l: list):
    # yield successivement l[0], l[1], l[2]
    yield from l

g = yield_from_example([1, 2, 3])

print(next(g))
print(next(g))
print(next(g))
print(next(g))



In [ ]:
# ---------------- RAPPEL sur ** ------------------------------

def func(p1, p2, p3):
    return p1, p2, p3

d = {"p1": 1, "p3": 3, "p2": 2}

print(func(d["p1"], d["p2"], d["p3"]))
print(func(**d))

In [ ]:
# ---------------------- fusion de dictionnaire ---------------------

def create_dico(defaults: dict={}, **opts):
    # fusion 1
    # for k, v in opts.items():
    #     defaults[k] = v
    # fusion2 avec .update()
    defaults.update(opts)
    return defaults

defaults = {"firstname": "matt", "lastname": "lamam"}

print(create_dico(defaults=defaults))
print(create_dico(defaults={"firstname": "matt", "lastname": "lamam"}, age=33, taille=1.75))
print(create_dico(defaults={"firstname": "matt", "lastname": "lamam"}, firstname='jean'))

# idem sans la fonction: on injecte les clés/valeurs dans un dico vide
opts = {"age": 33, "taille": 1.75, "firstname": "jean"}
new_dico = { **defaults, **opts }
new_dico



{'firstname': 'matt', 'lastname': 'lamam'}
{'firstname': 'matt', 'lastname': 'lamam', 'age': 33, 'taille': 1.75}
{'firstname': 'jean', 'lastname': 'lamam'}


{'firstname': 'jean', 'lastname': 'lamam', 'age': 33, 'taille': 1.75}

In [7]:
# ------------ paginer avec un générateur et un itérateur -----------------
import requests
from collections.abc import Iterator
from typing import Any
from itertools import islice


def iterer(session: requests.Session, url: str, params: dict ={},
           taille: int = 5) -> Iterator[dict[str, Any]]:
    params = {**params, "page[size]": taille}
    while True:
        r = session.get(url, params=params, timeout=(3.05, 10))
        r.raise_for_status()
        corps = r.json()
        yield from corps["data"]
        if not corps["meta"]["has_more"]:
            return
        params["page[after]"] = corps["meta"]["next_cursor"]


if __name__ == "__main__":
    # data = []
    URL = "https://gorest.co.in/public/v3/users"
    with requests.Session() as s:
        # for obj in iterer(s, URL):
        #     data.append(obj)

        ## version iterateur par ex. islice qui s'arrêtera sûrement à 100
        print(list(islice(iterer(s, URL),100)))

    # print(len(data))

[{'created_at': '2026-03-18T11:18:00Z', 'email': 'shaurya.verma@example.com', 'id': '019d00aa-b240-7d95-8cc2-479f04f365d7', 'name': 'Shaurya Verma', 'read_only': True, 'status': 'active', 'updated_at': '2026-03-18T11:18:00Z'}, {'created_at': '2026-03-15T11:01:00Z', 'email': 'pari.saxena@example.com', 'id': '019cf128-0de0-7828-8d88-95592d1ea61a', 'name': 'Pari Saxena', 'read_only': True, 'status': 'active', 'updated_at': '2026-03-15T11:01:00Z'}, {'created_at': '2026-03-12T10:44:00Z', 'email': 'krish.agarwal@example.com', 'id': '019ce1a5-6980-7f2e-8381-e95f5abbcbcd', 'name': 'Krish Agarwal', 'read_only': True, 'status': 'inactive', 'updated_at': '2026-03-12T10:44:00Z'}, {'created_at': '2026-03-09T10:27:00Z', 'email': 'navya.hegde@example.com', 'id': '019cd222-c520-7680-8714-2af748031197', 'name': 'Navya Hegde', 'read_only': True, 'status': 'active', 'updated_at': '2026-03-09T10:27:00Z'}, {'created_at': '2026-03-06T10:10:00Z', 'email': 'ayaan.khan@example.com', 'id': '019cc2a0-20c0-7551-8